# Combine Woody Cover + Fire Metrics — Preprocessing Step 3

**Version:** 1.0 (2026-03-09) © R. Butzer

## What this notebook does
Concatenates the reprojected Woody Cover raster and the MBA Fire Metrics mosaic
into a single multi-band GeoTIFF with named band descriptions.

## Inputs
- `_Runs\02_Woody_Cover\woody_cover_500m_3035.tif` — from `02_reproject_woodycover.ipynb`
- `_Runs\01_Fire_Metrics\MBA_500m_3035_DOY_season\MBA_burned_metrics_yearly_season_500m_mosaic.tif` — from `01_export_fire_metrics.ipynb`

## Outputs
- `_Runs\03_Woody_Fire_combined\woody_burned_MBA_full_combined.tif`
  - 353 bands: 41 Woody Cover (1985–2025) + 312 MBA metrics (26 years × 12 metrics)
  - → **Input for `04_ecoregions_MBA_analysis.ipynb` and `05_landcover_analysis.ipynb`**

In [ ]:
import rasterio
import numpy as np
import os
import time

workDir = r"\\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis"

# AKTUALISIERTE Pfade für den neuen MBA-Datensatz
mba_mosaic_path = os.path.join(
    workDir,
    r"_Runs\01_Fire_Metrics\MBA_500m_3035_DOY_season\MBA_burned_metrics_yearly_season_500m_mosaic.tif"
)
out_woody_path = os.path.join(
    workDir,
    r"_Runs\02_Woody_Cover\woody_cover_500m_3035.tif"
)
out_dir = os.path.join(workDir, "_Runs", "03_Woody_Fire_combined")
out_path = os.path.join(out_dir, "woody_burned_MBA_full_combined.tif")

# WICHTIG: Erstelle Zielordner falls nicht vorhanden
os.makedirs(out_dir, exist_ok=True)
print(f"✓ Zielordner: {out_dir}")

years_woody = list(range(1985, 2026))  # 41 Jahre
years_burned = list(range(2000, 2026))  # 26 Jahre
nodata_value = 255

print("\nLade Woody Cover Raster ...")
start = time.time()
with rasterio.open(out_woody_path) as woody_src:
    woody = woody_src.read()
    meta = woody_src.meta.copy()
    actual_woody_bands = woody.shape[0]
    years_woody_actual = list(range(1985, 1985 + actual_woody_bands))
print(f"✓ Woody Cover geladen. Dauer: {time.time() - start:.2f} Sekunden")
print(f"  Shape: {woody.shape} ({actual_woody_bands} Bänder)")

print("\nLade MBA Mosaic Raster (mit allen Metriken) ...")
start = time.time()
with rasterio.open(mba_mosaic_path) as mba_src:
    mba_full = mba_src.read()
    actual_mba_bands = mba_full.shape[0]
print(f"✓ MBA Mosaic geladen. Dauer: {time.time() - start:.2f} Sekunden")
print(f"  Shape: {mba_full.shape} ({actual_mba_bands} Bänder)")
print(f"  Erwartete Struktur: 26 Jahre × 12 Metriken = 312 Bänder")

# Bandstruktur definieren (12 Metriken pro Jahr)
metrics_per_year = 12
band_structure = [
    "burned",
    "count", 
    "doy_1",
    "doy_2",
    "doy_3",
    "doy_4",
    "doy_min",
    "doy_max",
    "doy_span",
    "month_first",
    "month_last",
    "season_length"
]

# Kombiniere die Daten: erst woody, dann alle MBA-Metriken
print("\nKombiniere Datensätze ...")
combined = np.concatenate([woody, mba_full], axis=0)
total_bands = combined.shape[0]

print(f"✓ Kombinierter Datensatz erstellt:")
print(f"  Woody Cover: {actual_woody_bands} Bänder")
print(f"  MBA Metriken: {actual_mba_bands} Bänder")
print(f"  Gesamt: {total_bands} Bänder")
print(f"  Shape: {combined.shape}")

# Metadaten aktualisieren (WICHTIG: BIGTIFF für große Dateien!)
meta.update({
    'nodata': nodata_value, 
    'count': total_bands,
    'compress': 'lzw',
    'tiled': True,
    'blockxsize': 512,
    'blockysize': 512,
    'BIGTIFF': 'YES'  # WICHTIG für Dateien > 4 GB!
})

print(f"\nSpeichere kombinierten Raster nach {out_path} ...")
print("(Dies kann einige Minuten dauern...)")
save_start = time.time()

with rasterio.open(out_path, "w", **meta) as dst:
    dst.write(combined)
    
save_duration = time.time() - save_start
print(f"✓ Kombinierter Raster gespeichert. Dauer: {save_duration:.2f} Sekunden")

# Setze detaillierte Bandbeschreibungen
print("\nSetze Bandbeschreibungen ...")
with rasterio.open(out_path, "r+") as dst:
    # Woody cover bands
    for i, year in enumerate(years_woody_actual):
        dst.set_band_description(i + 1, f"Woody_Cover_{year}")
    
    # MBA metric bands (12 pro Jahr)
    band_offset = actual_woody_bands
    for year_idx, year in enumerate(years_burned):
        for metric_idx, metric_name in enumerate(band_structure):
            band_num = band_offset + (year_idx * metrics_per_year) + metric_idx + 1
            dst.set_band_description(band_num, f"MBA_{year}_{metric_name}")

print("✓ Bandbeschreibungen gesetzt.")

# Ausgabe der Bandstruktur
print("\n" + "="*70)
print("BANDSTRUKTUR DES KOMBINIERTEN DATENSATZES")
print("="*70)

print(f"\n[Bänder 1-{actual_woody_bands}] Woody Cover:")
for i, year in enumerate(years_woody_actual[:3]):
    print(f"  Band {i+1}: Woody_Cover_{year}")
print("  ...")
for i, year in enumerate(years_woody_actual[-2:], start=len(years_woody_actual)-2):
    print(f"  Band {i+1}: Woody_Cover_{year}")

print(f"\n[Bänder {actual_woody_bands+1}-{total_bands}] MBA Metriken:")
print(f"  Struktur: 26 Jahre × 12 Metriken = {actual_mba_bands} Bänder")
print(f"\n  Metriken pro Jahr:")
for idx, metric in enumerate(band_structure, 1):
    print(f"    {idx:2d}. {metric}")

print(f"\n  Beispiel Jahr 2000 (Bänder {actual_woody_bands+1}-{actual_woody_bands+12}):")
for metric_idx, metric in enumerate(band_structure):
    band_num = actual_woody_bands + metric_idx + 1
    print(f"    Band {band_num}: MBA_2000_{metric}")

print(f"\n  Beispiel Jahr 2025 (Bänder {total_bands-11}-{total_bands}):")
for metric_idx, metric in enumerate(band_structure):
    band_num = total_bands - 11 + metric_idx
    print(f"    Band {band_num}: MBA_2025_{metric}")

# Dateigröße ausgeben
file_size_gb = os.path.getsize(out_path) / (1024**3)

print("\n" + "="*70)
print(f"✓✓✓ KOMBINIERTER DATENSATZ ERFOLGREICH ERSTELLT! ✓✓✓")
print("="*70)
print(f"  Ausgabedatei: {out_path}")
print(f"  Gesamtgröße: {file_size_gb:.2f} GB")
print(f"  Anzahl Bänder: {total_bands}")
print(f"  Rasterauflösung: {meta['height']} × {meta['width']} Pixel")
print(f"  Datentyp: {meta['dtype']}")
print(f"  NoData-Wert: {meta['nodata']}")
print(f"  Kompression: {meta.get('compress', 'keine')}")
print("="*70)

✓ Zielordner: \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\03_Woody_Fire_combined

Lade Woody Cover Raster ...
✓ Woody Cover geladen. Dauer: 22.92 Sekunden
  Shape: (40, 9660, 10483) (40 Bänder)

Lade MBA Mosaic Raster (mit allen Metriken) ...
✓ MBA Mosaic geladen. Dauer: 376.06 Sekunden
  Shape: (312, 9660, 10483) (312 Bänder)
  Erwartete Struktur: 26 Jahre × 12 Metriken = 312 Bänder

Kombiniere Datensätze ...
✓ Kombinierter Datensatz erstellt:
  Woody Cover: 40 Bänder
  MBA Metriken: 312 Bänder
  Gesamt: 352 Bänder
  Shape: (352, 9660, 10483)

Speichere kombinierten Raster nach \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\03_Woody_Fire_combined\woody_burned_MBA_full_combined.tif ...
(Dies kann einige Minuten dauern...)
✓ Kombinierter Raster gespeichert. Dauer: 763.01 Sekunden

Setze Bandbeschreibungen ...
✓ Bandbeschreibungen gesetzt.

BANDSTRUKTUR DES KOMBINIERTEN DATENSATZES

[Bänder 1-40] Woody Cover:
  Band 1: Woody_Cover_1985
  Band 2: Woody_Cover_1986
  Band